In [1]:
import os
from finetuning_data_formatter import TrainingExample, Guidelines, Category, LlamaGuardPromptConfigs, LlamaGuardGenerationConfigs, ExplanationPosition, AugmentationConfigs, FormatterConfigs, create_formatted_finetuning_examples
import csv
import numpy as np
import pandas as pd
from datasets import Dataset
import torch
import torch.nn as nn
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
from tqdm import tqdm
import random
import ast
tqdm.pandas()
import warnings
warnings.filterwarnings('ignore')

#seed set
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)



/home/yujin/anaconda3/envs/llama/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("CUDA_VISIBLE_DEVICES:", os.getenv("CUDA_VISIBLE_DEVICES"))

model_name = "meta-llama/Llama-Guard-3-1B"
#Llama Guard3 1B: "meta-llama/Llama-Guard-3-1B"
#Llama Guard2 8B: "meta-llama/Meta-Llama-Guard-2-8B"
#Llama Guard3 8B: "meta-llama/Llama-Guard-3-8B"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)


CUDA_VISIBLE_DEVICES: 4


In [3]:
train_prompt = pd.read_csv('data/train_prompt.csv')
test_prompt = pd.read_csv('test_prompt.csv')

In [4]:
train_prompt = train_prompt[train_prompt['unsafe'] == 'unsafe']
test_prompt = test_prompt[test_prompt['unsafe'] == 'unsafe']

In [6]:
training_examples = []
for index, row in train_prompt.iterrows():
    training_examples.append(TrainingExample(
        prompt=row['Prompt'],
        response='N/A', #row['Response']
        violated_category_codes=[],
        label=row['unsafe'],
        explanation="The prompt asks for personal health information."
    ))


In [7]:
guidelines = Guidelines(
    categories=[
        Category(name="Personal Health Information",
                 description="Information that can identify an individual health condition")
    ],
    category_code_prefix="O"
)

llama_guard_prompt_configs = LlamaGuardPromptConfigs(
    instructions_format_string="""[INST] Task: Check if there is unsafe content in 'User' messages in conversations according our safety policy with the below categories.

<BEGIN UNSAFE CONTENT CATEGORIES>
{guidelines}
<END UNSAFE CONTENT CATEGORIES>

<BEGIN CONVERSATION>

{conversation}

<END CONVERSATION>

Provide your safety assessment for $agent_type in the above conversation:
- First line must read 'safe' or 'unsafe'.
- If unsafe, a second line must include a comma-separated list of violated categories. [/INST]""",
    should_include_category_descriptions=True,
    should_shuffle_category_codes=True
)

llama_guard_generation_configs = LlamaGuardGenerationConfigs(
    should_list_violated_codes=True,
    explanation_position=ExplanationPosition.AFTER_DECISION
)

augmentation_configs = AugmentationConfigs(
    should_add_examples_with_dropped_nonviolated_prompt_categories=True,
    should_add_examples_with_dropped_violated_and_nonviolated_prompt_categories=True,
    explanation_for_augmentation_with_dropped_violated_and_nonviolated_prompt_categories="Response is safe because any riskiness in it is related to categories we're not trying to detect."
)

formatter_configs = FormatterConfigs(
    guidelines=guidelines,
    llama_guard_prompt_configs=llama_guard_prompt_configs,
    llama_guard_generation_configs=llama_guard_generation_configs,
    augmentation_configs=augmentation_configs,
    random_seed=42
)


In [ ]:
formatted_examples = create_formatted_finetuning_examples(
    training_examples, formatter_configs)

formatted_examples = list(set(formatted_examples))

In [ ]:
for param in model.parameters(): # model.generate -> RuntimeError: expected scalar type Float but found Half
    # Check if parameter dtype is  Half (float16)
        if param.dtype == torch.float16:
            param.data = param.data.to(torch.float32)
            
model_output_path = os.path.join('llama3_1b_finetuned')

training_args=TrainingArguments(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    report_to = "wandb",
    warmup_steps = 100,
    max_steps = 80,
    learning_rate = 0.00001,
    lr_scheduler_type = "constant", #add
    #fp16 = True,
    bf16=True,
    logging_steps = 1,
    output_dir = model_output_path
)

model.config.use_cache = False

def formatting_prompts_func(example):
      prompt = example['prompt']
      output_list = []
      for i in range(len(example['prompt'])):
        output_list.append(prompt[i])
      
      return output_list

formatted_examples = pd.DataFrame({'prompt': formatted_examples})
formatted_examples = Dataset.from_pandas(formatted_examples)

trainer = SFTTrainer(
        model = model,
        args = training_args,
        max_seq_length = min(2048, tokenizer.model_max_length),
        train_dataset = formatted_examples,
        #eval_dataset = test_dataset,
        formatting_func = formatting_prompts_func,
        #data_collator =collator,
        #peft_config=peft_config,
    )


In [ ]:
trainer.train()
trainer.save_model(model_output_path)

In [12]:
# Relevant conversation
conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "text", 
                "text": "What is the color of the sky?"
            },
        ],
    },
]

In [ ]:
# Relevant conversation
name = 'random name'
#conversations_8b
test_conversations = []
for prompt in test_prompt['Prompt']:
    prompt = prompt.format(name=name)   
    print(prompt)
    content = [{"type": "text", "text": prompt}]
    test_conversations.append([{
        "role": "user",
        "content": prompt
            },
        ])
    

def llama_guard_text_test(tokenizer, model, prompt, categories: dict[str, str]=None, excluded_category_keys: list[str]=[]):

    if categories is not None:
        input_ids = tokenizer.apply_chat_template(prompt, return_tensors="pt", categories=categories, excluded_category_keys=excluded_category_keys).to("cuda")
    else:
        input_ids = tokenizer.apply_chat_template(prompt, return_tensors="pt", excluded_category_keys=excluded_category_keys).to("cuda")
    input_prompt = tokenizer.decode(input_ids[0], skip_special_tokens=False)
    
    
    prompt_len = input_ids.shape[1]
    output = model.generate(
        input_ids=input_ids,
        max_new_tokens=20,
        output_scores=True,
        return_dict_in_generate=True,
        pad_token_id=0,
    )
    generated_tokens = output.sequences[:, prompt_len:]
    
    response = tokenizer.decode(
        generated_tokens[0], skip_special_tokens=False
    )
    return input_prompt, response


In [14]:

def llama_guard_text_test(tokenizer, model, prompt, categories: dict[str, str]=None, excluded_category_keys: list[str]=[]):

    if categories is not None:
        input_ids = tokenizer.apply_chat_template(prompt, return_tensors="pt", categories=categories, excluded_category_keys=excluded_category_keys).to("cuda")
    else:
        input_ids = tokenizer.apply_chat_template(prompt, return_tensors="pt", excluded_category_keys=excluded_category_keys).to("cuda")
    input_prompt = tokenizer.decode(input_ids[0], skip_special_tokens=False)
    
    
    prompt_len = input_ids.shape[1]
    output = model.generate(
        input_ids=input_ids,
        max_new_tokens=20,
        output_scores=True,
        return_dict_in_generate=True,
        pad_token_id=0,
    )
    generated_tokens = output.sequences[:, prompt_len:]
    
    response = tokenizer.decode(
        generated_tokens[0], skip_special_tokens=False
    )
    return input_prompt, response

In [15]:
lg_mm_model_id = "meta-llama/Llama-Guard-3-1B"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
clinical_finetuned_tokenizer = AutoTokenizer.from_pretrained('llama3_1b_finetuned_max_step_80')
clinical_finetuned_model = AutoModelForCausalLM.from_pretrained('llama3_1b_finetuned_max_step_80', torch_dtype=torch.bfloat16).to(device)


In [ ]:
results = []
for conversation in test_conversations:
    print(conversation[0]['content'])
    decoded_input_prompt, response = llama_guard_text_test(clinical_finetuned_tokenizer, clinical_finetuned_model, conversation)
    response = response.replace("<|eot_id|>", "")
    print(response)

    results.append({
        "content": conversation[0]['content'],
        "response": response
    })

In [17]:
df_results = pd.DataFrame(results)
df_results['unsafe'] = df_results['response'].apply(lambda x: True if 'unsafe' in x.lower() else False)
unsafe_count = df_results['unsafe'].mean() 
print(unsafe_count)

0.445


In [ ]:
df_wikidocs = pd.read_csv('/home/yujin/source/temp/llama_guard/data/medical_meadow_wikidoc.csv')
df_wikidocs.rename(columns={'0': 'Prompt'}, inplace=True)

test_wikidocs = []
for prompt in df_wikidocs['Prompt']:   
    print(prompt)
    content = [{"type": "text", "text": prompt}]
    test_wikidocs.append([{
        "role": "user",
        "content": prompt
            },
        ])
    

# Original LLaMA Guard Performance

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, MllamaForConditionalGeneration, AutoProcessor, MllamaProcessor, GenerationConfig
from typing import List, Any
import torch

original_model = "meta-llama/Llama-Guard-3-1B"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
original_tokenizer = AutoTokenizer.from_pretrained(original_model)
original_model = AutoModelForCausalLM.from_pretrained(original_model, torch_dtype=torch.bfloat16).to(device)

In [ ]:
name = 'random name'

test_conversations = []
for prompt in test_prompt['Prompt']:
    prompt = prompt.format(name=name)   
    print(prompt)
    content = [{"type": "text", "text": prompt}]
    test_conversations.append([{
        "role": "user",
        "content": prompt
            },
        ])
    

def llama_guard_text_test(tokenizer, model, prompt, categories: dict[str, str]=None, excluded_category_keys: list[str]=[]):

    if categories is not None:
        input_ids = tokenizer.apply_chat_template(prompt, return_tensors="pt", categories=categories, excluded_category_keys=excluded_category_keys).to("cuda")
    else:
        input_ids = tokenizer.apply_chat_template(prompt, return_tensors="pt", excluded_category_keys=excluded_category_keys).to("cuda")
    input_prompt = tokenizer.decode(input_ids[0], skip_special_tokens=False)
    
    
    prompt_len = input_ids.shape[1]
    output = model.generate(
        input_ids=input_ids,
        max_new_tokens=20,
        output_scores=True,
        return_dict_in_generate=True,
        pad_token_id=0,
    )
    generated_tokens = output.sequences[:, prompt_len:]
    
    response = tokenizer.decode(
        generated_tokens[0], skip_special_tokens=False
    )
    return input_prompt, response


In [ ]:
original_results = []
for conversation in test_conversations:
    decoded_input_prompt, response = llama_guard_text_test(original_tokenizer, original_model, conversation)
    response = response.replace("<|eot_id|>", "")
    #print(response)

    original_results.append({
        "content": conversation[0]['content'],
        "response": response
    })


df_original_results = pd.DataFrame(original_results)
df_original_results['unsafe'] = df_original_results['response'].apply(lambda x: True if 'unsafe' in x.lower() else False)
unsafe_count = df_original_results['unsafe'].mean() 
print(unsafe_count)